In [1]:
import glob
import os
import re
import sys

import numpy as np
import optuna
import pandas as pd

In [2]:
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

from drGAT.metrics import get_parsed_df

In [3]:
def extract_method_data(filename):
    match = re.match(r"([A-Za-zA-Z0-9]+)_([a-z0-9]+)\.sqlite3", filename)
    return match.groups() if match else ("unknown", "unknown")

In [13]:
all_dfs = []

for file in glob.glob("*.sqlite3"):
    method, data = extract_method_data(os.path.basename(file))
    try:
        study = optuna.load_study(study_name=method, storage=f"sqlite:///{file}")
        df_all = study.trials_dataframe()

        # ✅ COMPLETE 状態の数を表示
        n_complete = (df_all["state"] == "COMPLETE").sum()
        print(f"✅ {file}: {n_complete} trials completed")

        df_valid = df_all.dropna(
            subset=["values_0", "values_1", "values_2", "values_3"]
        )

        if df_valid.shape[0] == 0:
            continue

        # user_attrs から評価指標を取り出す
        df_metrics = df_valid.iloc[:, 20:-2]
        df_metrics.columns = [c.replace("user_attrs_", "") for c in df_metrics.columns]
        parsed_df = get_parsed_df(df_metrics)

        # ハイパーパラメータ列だけ抽出
        df_params = df_valid[
            [c for c in df_valid.columns if "params" in c]
        ].reset_index(drop=True)
        parsed_df = pd.concat([parsed_df.reset_index(drop=True), df_params], axis=1)

        # method / data 列を追加
        parsed_df["method"] = method
        parsed_df["data"] = data

        all_dfs.append(parsed_df)

    except Exception as e:
        print(f"❌ Failed to parse {file}: {e}")

if all_dfs:
    summary_df = pd.concat(all_dfs, ignore_index=True)
    summary_df["AUPR_num"] = summary_df["AUPR"].str.extract(r"([\d\.]+)").astype(float)

    # 最良インデックス取得
    best_idxs = summary_df.groupby(["data", "method"])["AUPR_num"].idxmax()
    best_df = summary_df.loc[best_idxs].drop(columns=["AUPR_num"])

    # ✅ インデックス設定: method, data の順で MultiIndex
    best_df.set_index(
        [
            "data",
            "method",
        ],
        inplace=True,
    )

    # ✅ data をキーに並べ替え（index の level 1 をソート）
    best_df.sort_index(level="data", inplace=True)

    # 任意の順に並べたいカラム
    desired_order = ["ACC", "Precision", "Recall", "F1", "AUROC", "AUPR"]

    # その順に並び替える（残りのカラムはそのまま後ろへ）
    other_cols = [col for col in best_df.columns if col not in desired_order]
    best_df = best_df[desired_order + other_cols]

    # インデックスの data レベルを並び替え
    desired_data_order = [
        "nci",
        "gdsc1",
        "gdsc2",
        "ctrp",
    ]  # 小文字なら小文字にそろえる！

    # sort_index ではなく reindex で順序を明示
    best_df = best_df.reindex(
        best_df.index.reindex(
            sorted(best_df.index, key=lambda x: desired_data_order.index(x[0]))
        )[0]
    )

    display(best_df)
else:
    print("No valid studies found.")

✅ GAT_ctrp.sqlite3: 1 trials completed
✅ GAT_gdsc1.sqlite3: 3 trials completed
✅ GAT_gdsc2.sqlite3: 1 trials completed
✅ GAT_nci.sqlite3: 3 trials completed
✅ Transformer_gdsc1.sqlite3: 1 trials completed
✅ Transformer_ctrp.sqlite3: 1 trials completed
✅ Transformer_nci.sqlite3: 1 trials completed
✅ Transformer_gdsc2.sqlite3: 3 trials completed
✅ GATv2_gdsc1.sqlite3: 2 trials completed
✅ GATv2_ctrp.sqlite3: 3 trials completed
✅ GATv2_gdsc2.sqlite3: 3 trials completed
✅ GATv2_nci.sqlite3: 2 trials completed


ACC        Precision           Recall  \
data  method                                                           
nci   GAT          0.823 (± 0.027)   0.79 (± 0.039)  0.877 (± 0.032)   
      GATv2        0.855 (± 0.007)    0.86 (± 0.01)  0.842 (± 0.025)   
      Transformer  0.849 (± 0.007)   0.85 (± 0.011)  0.842 (± 0.025)   
gdsc1 GAT          0.594 (± 0.026)   0.62 (± 0.057)  0.478 (± 0.186)   
      GATv2         0.732 (± 0.02)   0.76 (± 0.034)  0.667 (± 0.101)   
      Transformer  0.558 (± 0.015)  0.565 (± 0.012)  0.417 (± 0.191)   
gdsc2 GAT          0.693 (± 0.089)  0.814 (± 0.105)  0.589 (± 0.294)   
      GATv2        0.758 (± 0.066)  0.761 (± 0.077)   0.786 (± 0.03)   
      Transformer  0.737 (± 0.106)  0.728 (± 0.109)  0.858 (± 0.088)   
ctrp  GAT          0.766 (± 0.067)  0.775 (± 0.072)  0.781 (± 0.054)   
      GATv2        0.664 (± 0.007)  0.661 (± 0.021)  0.733 (± 0.047)   
      Transformer  0.831 (± 0.006)  0.855 (± 0.006)  0.814 (± 0.009)   

                                F1            AUROC             AUPR  \
data  method                                                           
nci   GAT           0.83 (± 0.022)  0.895 (± 0.012)  0.882 (± 0.014)   
      GATv2         0.851 (± 0.01)  0.914 (± 0.004)  0.897 (± 0.009)   
      Transformer  0.846 (± 0.009)  0.909 (± 0.004)  0.893 (± 0.006)   
gdsc1 GAT          0.519 (± 0.102)  0.658 (± 0.029)  0.645 (± 0.036)   
      GATv2        0.705 (± 0.051)  0.813 (± 0.008)  0.804 (± 0.007)   
      Transformer  0.458 (± 0.137)  0.593 (± 0.013)   0.57 (± 0.021)   
gdsc2 GAT           0.627 (± 0.22)  0.835 (± 0.019)  0.831 (± 0.024)   
      GATv2        0.772 (± 0.053)  0.827 (± 0.068)  0.821 (± 0.071)   
      Transformer  0.778 (± 0.043)  0.861 (± 0.011)  0.857 (± 0.014)   
ctrp  GAT          0.777 (± 0.059)   0.84 (± 0.064)  0.852 (± 0.053)   
      GATv2        0.694 (± 0.008)  0.744 (± 0.004)  0.775 (± 0.008)   
      Transformer  0.834 (± 0.005)  0.906 (± 0.003)   0.91 (± 0.004)   

                      Balanced_ACC  params_T_max params_activation  \
data  method                                                         
nci   GAT          0.824 (± 0.027)           NaN              gelu   
      GATv2        0.854 (± 0.008)         558.0              relu   
      Transformer  0.849 (± 0.007)           NaN              gelu   
gdsc1 GAT          0.593 (± 0.028)           NaN              relu   
      GATv2        0.731 (± 0.022)        3378.0              relu   
      Transformer   0.553 (± 0.02)           NaN              gelu   
gdsc2 GAT          0.699 (± 0.083)        1356.0              gelu   
      GATv2        0.757 (± 0.066)         888.0              gelu   
      Transformer   0.729 (± 0.12)           NaN              gelu   
ctrp  GAT          0.765 (± 0.067)         462.0              gelu   
      GATv2        0.661 (± 0.009)        2109.0              relu   
      Transformer  0.832 (± 0.005)        2497.0              gelu   

                   params_dropout1  ...  params_heads  params_hidden1  \
data  method                        ...                                 
nci   GAT                     0.50  ...           4.0           873.0   
      GATv2                   0.15  ...           8.0           927.0   
      Transformer             0.10  ...           2.0           728.0   
gdsc1 GAT                     0.25  ...           2.0           268.0   
      GATv2                   0.25  ...           7.0           278.0   
      Transformer             0.45  ...           4.0           484.0   
gdsc2 GAT                     0.40  ...           6.0           525.0   
      GATv2                   0.40  ...           5.0           690.0   
      Transformer             0.50  ...           2.0           753.0   
ctrp  GAT                     0.10  ...           3.0           517.0   
      GATv2                   0.45  ...           3.0           297.0   
      Transformer             0.30  ...           2.0           995.0   

         

In [14]:
best_df[desired_order]

ACC        Precision           Recall  \
data  method                                                           
nci   GAT          0.823 (± 0.027)   0.79 (± 0.039)  0.877 (± 0.032)   
      GATv2        0.855 (± 0.007)    0.86 (± 0.01)  0.842 (± 0.025)   
      Transformer  0.849 (± 0.007)   0.85 (± 0.011)  0.842 (± 0.025)   
gdsc1 GAT          0.594 (± 0.026)   0.62 (± 0.057)  0.478 (± 0.186)   
      GATv2         0.732 (± 0.02)   0.76 (± 0.034)  0.667 (± 0.101)   
      Transformer  0.558 (± 0.015)  0.565 (± 0.012)  0.417 (± 0.191)   
gdsc2 GAT          0.693 (± 0.089)  0.814 (± 0.105)  0.589 (± 0.294)   
      GATv2        0.758 (± 0.066)  0.761 (± 0.077)   0.786 (± 0.03)   
      Transformer  0.737 (± 0.106)  0.728 (± 0.109)  0.858 (± 0.088)   
ctrp  GAT          0.766 (± 0.067)  0.775 (± 0.072)  0.781 (± 0.054)   
      GATv2        0.664 (± 0.007)  0.661 (± 0.021)  0.733 (± 0.047)   
      Transformer  0.831 (± 0.006)  0.855 (± 0.006)  0.814 (± 0.009)   

                                F1            AUROC             AUPR  
data  method                                                          
nci   GAT           0.83 (± 0.022)  0.895 (± 0.012)  0.882 (± 0.014)  
      GATv2         0.851 (± 0.01)  0.914 (± 0.004)  0.897 (± 0.009)  
      Transformer  0.846 (± 0.009)  0.909 (± 0.004)  0.893 (± 0.006)  
gdsc1 GAT          0.519 (± 0.102)  0.658 (± 0.029)  0.645 (± 0.036)  
      GATv2        0.705 (± 0.051)  0.813 (± 0.008)  0.804 (± 0.007)  
      Transformer  0.458 (± 0.137)  0.593 (± 0.013)   0.57 (± 0.021)  
gdsc2 GAT           0.627 (± 0.22)  0.835 (± 0.019)  0.831 (± 0.024)  
      GATv2        0.772 (± 0.053)  0.827 (± 0.068)  0.821 (± 0.071)  
      Transformer  0.778 (± 0.043)  0.861 (± 0.011)  0.857 (± 0.014)  
ctrp  GAT          0.777 (± 0.059)   0.84 (± 0.064)  0.852 (± 0.053)  
      GATv2        0.694 (± 0.008)  0.744 (± 0.004)  0.775 (± 0.008)  
      Transformer  0.834 (± 0.005)  0.906 (± 0.003)   0.91 (± 0.004)